# Carga de imagenes al datastore y mosaic dataset

Flujo operativo posterior a la preparacion del notebook `008`. Usa `04_ready_for_datastore.csv` para copiar archivos al datastore y `06_attribute_updates.csv` para actualizar atributos del mosaic dataset.

In [ ]:
from datetime import datetime
from pathlib import Path
import importlib

import pandas as pd

import core.mosaic_loader as mosaic_loader
mosaic_loader = importlib.reload(mosaic_loader)
from core.mosaic_loader import *

# PARAMETROS
RUN_PREPARACION = "20260615_155625"
OUTPUT_PREPARACION_DIR = Path.cwd() / "outputs" / "preparacion_carga_mosaico" / RUN_PREPARACION

READY_FOR_DATASTORE_CSV = OUTPUT_PREPARACION_DIR / "04_ready_for_datastore.csv"
ATTRIBUTE_UPDATES_CSV = OUTPUT_PREPARACION_DIR / "06_attribute_updates.csv"

PATH_MOSAIC_DATASET = r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proyectos_ArcGIS\APRX\CL MLP PAO Aereo Image Server_v2\SQLServer-amssclgis06_ArcGIS-Aereo.sde\OWD.CL_MLP_PAO_IF_Ortho_Geosupport"

# Seguridad operacional: validar primero con DRY_RUN=True. Cambiar a False para ejecutar copia/carga/update.
DRY_RUN = True
OVERWRITE_COPY = False
SKIP_EXISTING_MOSAIC_NAME = True

# Valores fijos definidos para esta carga.
MAXPS_VALUE = 10000
LOWPS_VALUE = 0.15

run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = Path.cwd() / "outputs" / "carga_mosaico" / run_timestamp
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Preparacion:", OUTPUT_PREPARACION_DIR)
print("CSV carga:", READY_FOR_DATASTORE_CSV)
print("CSV atributos:", ATTRIBUTE_UPDATES_CSV)
print("Mosaic dataset:", PATH_MOSAIC_DATASET)
print("Salida:", OUTPUT_DIR)
print("DRY_RUN:", DRY_RUN)

## 1. Cargar manifiestos

Se valida que cada imagen lista tenga atributos asociados antes de ejecutar cualquier operacion.

In [ ]:
load_df = load_ready_and_attributes(READY_FOR_DATASTORE_CSV, ATTRIBUTE_UPDATES_CSV)

required_columns = ["path", "destination_path", "Name", "Sector", "Fecha_Adqui", "URL", "Proyecto", "Sensor", "Fecha_Publ"]
missing_columns = [column for column in required_columns if column not in load_df.columns]
if missing_columns:
    raise ValueError(f"Faltan columnas requeridas: {missing_columns}")

validation_summary = pd.DataFrame(
    [
        {"metric": "records_to_process", "value": len(load_df)},
        {"metric": "missing_source_path", "value": int((~load_df["path"].map(lambda value: Path(value).exists())).sum())},
        {"metric": "missing_destination_path", "value": int(load_df["destination_path"].isna().sum())},
        {"metric": "unique_destination_paths", "value": int(load_df["destination_path"].nunique())},
        {"metric": "unique_names", "value": int(load_df["Name"].nunique())},
    ]
)

display(validation_summary)
display(load_df[["file_name", "Name", "destination_path", "Sector", "Fecha_Adqui", "Proyecto", "Sensor", "Fecha_Publ"]].head(20))

## 2. Ejecutar copia, carga al mosaico, footprints y atributos

Con `DRY_RUN=True` no escribe archivos ni modifica el mosaic dataset. Con `DRY_RUN=False` ejecuta el flujo completo por imagen.

In [ ]:
results = []

for index, row in load_df.iterrows():
    print(f"[{index + 1}/{len(load_df)}] {row['Name']}")
    result = process_mosaic_load_row(
        row,
        PATH_MOSAIC_DATASET,
        overwrite_copy=OVERWRITE_COPY,
        skip_existing_mosaic_name=SKIP_EXISTING_MOSAIC_NAME,
        maxps_value=MAXPS_VALUE,
        lowps_value=LOWPS_VALUE,
        dry_run=DRY_RUN,
    )
    results.append(result)

results_df = pd.DataFrame(results)
display(results_df)
display(results_df["overall_status"].value_counts(dropna=False).reset_index(name="count").rename(columns={"index": "overall_status"}))

## 3. Exportar resultados de ejecucion

In [ ]:
summary_rows = [
    {"metric": "run_timestamp", "value": run_timestamp},
    {"metric": "preparation_run", "value": RUN_PREPARACION},
    {"metric": "dry_run", "value": DRY_RUN},
    {"metric": "mosaic_dataset", "value": PATH_MOSAIC_DATASET},
    {"metric": "records_to_process", "value": len(load_df)},
    {"metric": "maxps_value", "value": MAXPS_VALUE},
    {"metric": "lowps_value", "value": LOWPS_VALUE},
]

for column in ["copy_status", "mosaic_add_status", "footprint_status", "attribute_status", "overall_status"]:
    if column in results_df.columns:
        for status, count in results_df[column].value_counts(dropna=False).items():
            summary_rows.append({"metric": f"{column}_{status}", "value": int(count)})

summary_df = pd.DataFrame(summary_rows)

summary_csv = OUTPUT_DIR / "00_summary.csv"
results_csv = OUTPUT_DIR / "01_load_results.csv"
errors_csv = OUTPUT_DIR / "02_errors.csv"

summary_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")
results_df.to_csv(results_csv, index=False, encoding="utf-8-sig")
error_columns = [column for column in results_df.columns if column.endswith("_error")]
error_filter = results_df[error_columns].notna().any(axis=1) if error_columns else pd.Series(False, index=results_df.index)
results_df[error_filter].to_csv(errors_csv, index=False, encoding="utf-8-sig")

display(summary_df)
print("Resultados exportados en:", OUTPUT_DIR)